# Zaskaleta Clone v1 — Full-Length Talking Test

**Runtime → Run all.**

Тест використовує `MASTER_BEHAVIOR_01.mp4` повністю, без скорочення тривалості. Перевіряємо стабільність обличчя, природність lip-sync, голос і поведінку на реальному відеореференсі.


In [ ]:
import os, re, subprocess, sys
from pathlib import Path
import torch

SOURCE_FOLDER_ID='13Wye5lVZPOcUryXbXhplad_FkxM7lG4a'
SOURCE_LOCAL=Path('/content/zaskaleta_fixed_source')
ROOT=Path('/content/zaskaleta-ai-twin-colab')

if not torch.cuda.is_available():
    print('⛔ GPU недоступний. Увімкни T4 і запусти ще раз.')
    raise SystemExit(0)
print('✅ CUDA/T4 доступний')

if ROOT.exists():
    subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git',str(ROOT)],check=True)
WORKER=ROOT/'worker'

from google.colab import auth
print('\n🔐 Авторизація Google Drive')
auth.authenticate_user()
try:
    import googleapiclient.discovery
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','google-api-python-client','google-auth','google-auth-httplib2'],check=True)

if SOURCE_LOCAL.exists():
    subprocess.run(['rm','-rf',str(SOURCE_LOCAL)],check=True)
SOURCE_LOCAL.mkdir(parents=True,exist_ok=True)
print('\n📥 Завантажую SOURCE...')
subprocess.run([sys.executable,str(WORKER/'fixed_drive_folder_sync.py'),'pull','--folder-id',SOURCE_FOLDER_ID,'--local-dir',str(SOURCE_LOCAL)],check=True)

behavior=list(SOURCE_LOCAL.rglob('MASTER_BEHAVIOR_01.mp4'))
if not behavior:
    raise RuntimeError('MASTER_BEHAVIOR_01.mp4 не знайдено у SOURCE')
print('✅ Behavior reference:',behavior[0])

env=os.environ.copy()
env['APP_DIR']=str(WORKER)
env['MUSETALK_ROOT']='/content/MuseTalk'
env['VENV_DIR']='/content/ai-twin-py311'
env['PRIMARY_DRIVE']=str(SOURCE_LOCAL)

print('\n========== INSTALLER START ==========')
iproc=subprocess.Popen(['bash',str(WORKER/'install_gpu_engines.sh')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iproc.stdout:
    print(line,end='')
icode=iproc.wait()
if icode != 0:
    raise RuntimeError(f'Installer failed: {icode}')
print('========== INSTALLER OK ==========')

PY='/content/ai-twin-py311/bin/python'
cmd=[PY,str(WORKER/'run_clone_v1_test.py'),'--root',str(ROOT),'--mydrive',str(SOURCE_LOCAL)]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    print(line,end='')
    lines.append(line)
code=proc.wait()
if code != 0:
    raise RuntimeError('Clone v1 test failed\n'+''.join(lines[-80:]))
out=''.join(lines)
m=re.search(r'^FINAL_PATH=(.+)$',out,re.MULTILINE)
if not m:
    raise RuntimeError('FINAL_PATH not found')
FINAL=m.group(1).strip()

print('\n📤 Зберігаю тест назад у SOURCE...')
subprocess.run([sys.executable,str(WORKER/'fixed_drive_folder_sync.py'),'push','--folder-id',SOURCE_FOLDER_ID,'--local-dir',str(SOURCE_LOCAL)],check=True)
print('✅ CLONE V1 FULL TEST READY:',FINAL)
from IPython.display import Video, display
display(Video(FINAL,embed=True,width=360))
